In [25]:
import pandas as pd

df = pd.read_csv("/Users/moekhantzaw/DATA_ANALYST/for practical/churn_clean.csv")
print(df.shape)
df.head()

(7043, 22)


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn,churn_flag
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No,0
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,No,No,No,One year,No,Mailed check,56.95,1889.50,No,0
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes,1
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No,0
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes,1


In [27]:
print(df.dtypes)
print(df.isnull().sum())

customerID           object
gender               object
SeniorCitizen         int64
Partner              object
Dependents           object
tenure                int64
PhoneService         object
MultipleLines        object
InternetService      object
OnlineSecurity       object
OnlineBackup         object
DeviceProtection     object
TechSupport          object
StreamingTV          object
StreamingMovies      object
Contract             object
PaperlessBilling     object
PaymentMethod        object
MonthlyCharges      float64
TotalCharges        float64
Churn                object
churn_flag            int64
dtype: object
customerID           0
gender               0
SeniorCitizen        0
Partner              0
Dependents           0
tenure               0
PhoneService         0
MultipleLines        0
InternetService      0
OnlineSecurity       0
OnlineBackup         0
DeviceProtection     0
TechSupport          0
StreamingTV          0
StreamingMovies      0
Contract             0
Pa

In [31]:
print(df["churn_flag"].mean())

0.2653698707936959


In [33]:
for col in ["Contract", "PaymentMethod", "InternetService", "TechSupport", "OnlineSecurity", "PaperlessBilling"]:
    print(df.groupby(col)["churn_flag"].mean().sort_values(ascending=False))
    print()

Contract
Month-to-month    0.427097
One year          0.112695
Two year          0.028319
Name: churn_flag, dtype: float64

PaymentMethod
Electronic check             0.452854
Mailed check                 0.191067
Bank transfer (automatic)    0.167098
Credit card (automatic)      0.152431
Name: churn_flag, dtype: float64

InternetService
Fiber optic    0.418928
DSL            0.189591
No             0.074050
Name: churn_flag, dtype: float64

TechSupport
No                     0.416355
Yes                    0.151663
No internet service    0.074050
Name: churn_flag, dtype: float64

OnlineSecurity
No                     0.417667
Yes                    0.146112
No internet service    0.074050
Name: churn_flag, dtype: float64

PaperlessBilling
Yes    0.335651
No     0.163301
Name: churn_flag, dtype: float64



In [35]:
df["tenure_group"] = pd.cut(df["tenure"], bins=[-1, 12, 24, 48, 72],
                            labels=["0-12", "13-24", "25-48", "49-72"])
print(df.groupby("tenure_group")["churn_flag"].mean())

tenure_group
0-12     0.474382
13-24    0.287109
25-48    0.203890
49-72    0.095132
Name: churn_flag, dtype: float64


/var/folders/80/df4bbr5j4n3gr3hgn96x9q7r0000gn/T/ipykernel_20118/2158940340.py:3: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  print(df.groupby("tenure_group")["churn_flag"].mean())


In [39]:
print(df["MonthlyCharges"].head(10))

0     29.85
1     56.95
2     53.85
3     42.30
4     70.70
5     99.65
6     89.10
7     29.75
8    104.80
9     56.15
Name: MonthlyCharges, dtype: float64


In [41]:
lost = df[df["churn_flag"] == 1]["MonthlyCharges"].sum()
total = df["MonthlyCharges"].sum()
print("Monthly revenue lost to churn:", round(lost, 2))
print("Share of total monthly revenue:", round(lost / total, 4))

Monthly revenue lost to churn: 139130.85
Share of total monthly revenue: 0.305


In [43]:
# Does tech support matter within each contract type?
print(df.groupby(["Contract", "TechSupport"], observed=False)["churn_flag"].mean().unstack().round(3))

TechSupport        No  No internet service    Yes
Contract                                         
Month-to-month  0.504                0.189  0.307
One year        0.147                0.025  0.136
Two year        0.059                0.008  0.035


In [45]:
# Does contract matter within each tenure group?
print(df.groupby(["tenure_group", "Contract"], observed=False)["churn_flag"].mean().unstack().round(3))

Contract      Month-to-month  One year  Two year
tenure_group                                    
0-12                   0.514     0.105     0.000
13-24                  0.377     0.081     0.000
25-48                  0.329     0.106     0.022
49-72                  0.260     0.129     0.033


In [47]:
# Is fiber optic churn just an electronic check effect?
print(df.groupby(["InternetService", "PaymentMethod"], observed=False)["churn_flag"].mean().unstack().round(3))

PaymentMethod    Bank transfer (automatic)  Credit card (automatic)  \
InternetService                                                       
DSL                                  0.094                    0.121   
Fiber optic                          0.289                    0.253   
No                                   0.054                    0.027   

PaymentMethod    Electronic check  Mailed check  
InternetService                                  
DSL                         0.319         0.207  
Fiber optic                 0.532         0.426  
No                          0.123         0.096  


In [49]:
# Sample size behind the tenure x contract table
print(df.groupby(["tenure_group", "Contract"], observed=False).size().unstack())

# Where the lost revenue sits, by contract type
seg = df[df["churn_flag"] == 1].groupby("Contract")["MonthlyCharges"].sum().round(2)
print(seg)
print((seg / seg.sum()).round(3))

# The highest-risk group: month-to-month + fiber + electronic check
high = df[(df["Contract"] == "Month-to-month") &
          (df["InternetService"] == "Fiber optic") &
          (df["PaymentMethod"] == "Electronic check")]
print("customers:", len(high))
print("churn rate:", round(high["churn_flag"].mean(), 3))
print("monthly revenue lost:", round(high[high["churn_flag"] == 1]["MonthlyCharges"].sum(), 2))

Contract      Month-to-month  One year  Two year
tenure_group                                    
0-12                    1994       124        68
13-24                    737       197        90
25-48                    802       518       274
49-72                    342       634      1263
Contract
Month-to-month    120847.10
One year           14118.45
Two year            4165.30
Name: MonthlyCharges, dtype: float64
Contract
Month-to-month    0.869
One year          0.101
Two year          0.030
Name: MonthlyCharges, dtype: float64
customers: 1307
churn rate: 0.604
monthly revenue lost: 68281.5


In [51]:
seg_lost = 68281.50
for pct in [0.10, 0.20, 0.30]:
    monthly = seg_lost * pct
    print(f"{int(pct*100)}% reduction -> ${monthly:,.0f}/month, ${monthly*12:,.0f}/year")

10% reduction -> $6,828/month, $81,938/year
20% reduction -> $13,656/month, $163,876/year
30% reduction -> $20,484/month, $245,813/year


In [53]:
df["high_risk_segment"] = ((df["Contract"] == "Month-to-month") &
                           (df["InternetService"] == "Fiber optic") &
                           (df["PaymentMethod"] == "Electronic check")).map({True: "High risk", False: "Other"})
df.to_csv("/Users/moekhantzaw/DATA_ANALYST/for practical/churn_tableau.csv", index=False)